# 07 — Espacios Vectoriales

**Objetivo:** Entender los conceptos de independencia lineal, base, dimensión y rango que revelan la estructura de los datos de analytics.

## Contexto matemático

Un **espacio vectorial** $V$ es un conjunto cerrado bajo suma y multiplicación escalar. Un **subespacio** de $\mathbb{R}^n$ contiene al cero y es cerrado bajo las mismas operaciones.

Subespacios fundamentales de $A \in \mathbb{R}^{m\times n}$:

| Subespacio | Definición | Dimensión |
|---|---|---|
| Espacio columna $C(A)$ | $\{A\mathbf{x} : \mathbf{x}\in\mathbb{R}^n\}$ | $\text{rango}(A)$ |
| Espacio nulo $N(A)$ | $\{\mathbf{x} : A\mathbf{x}=\mathbf{0}\}$ | $n - \text{rango}(A)$ |
| Espacio fila $C(A^T)$ | $\{A^T\mathbf{y} : \mathbf{y}\in\mathbb{R}^m\}$ | $\text{rango}(A)$ |

**Teorema fundamental:** $\dim(C(A)) + \dim(N(A)) = n$ (Nullity-Rank Theorem).

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)
print('Setup listo')

## 1 — Independencia lineal

Un conjunto de vectores $\{\mathbf{v}_1, \ldots, \mathbf{v}_k\}$ es **linealmente independiente** si:

$$c_1\mathbf{v}_1 + c_2\mathbf{v}_2 + \cdots + c_k\mathbf{v}_k = \mathbf{0} \implies c_1 = c_2 = \cdots = c_k = 0$$

En la práctica: los vectores son LI si y solo si la matriz formada por ellos tiene rango = k.

In [ ]:
# ¿Son independientes estas 4 métricas de canal?
np.random.seed(0)
# Sessions, Clicks, Impressions, CTR(=Clicks/Impressions)
sesiones     = np.random.randint(100, 1000, 8).astype(float)
clicks       = np.random.randint(50, 500, 8).astype(float)
impresiones  = clicks * np.random.uniform(5, 15, 8)   # clicks/impr está implícito
ctr          = clicks / impresiones                   # ← combinación lineal de las otras dos

M = np.column_stack([sesiones, clicks, impresiones, ctr])
print(f'Shape M: {M.shape}')
print(f'Rango = {np.linalg.matrix_rank(M)}')
print(f'→ Solo {np.linalg.matrix_rank(M)} de las 4 métricas son verdaderamente independientes')

# Con solo 3 métricas independientes:
M3 = np.column_stack([sesiones, clicks, impresiones])
print(f'\nRango(sesiones, clicks, impresiones) = {np.linalg.matrix_rank(M3)}')
print('→ Las 3 son independientes entre sí')

## 2 — Rango: cuántas direcciones independientes hay

El **rango** de $A$ es el número de columnas (o filas) linealmente independientes. Revela cuánta información 'nueva' aporta cada columna.

- **Rango completo de columna** (full column rank): $\text{rango}(A) = n$ → sistema tiene solución única.
- **Rango deficiente** (rank-deficient): hay columnas redundantes → multicolinealidad.

In [ ]:
# Matriz funnel: ¿cuántas etapas son realmente independientes?
np.random.seed(42)
n_semanas = 12

# 5 métricas de canal, pero la última es combinación de las otras
base = np.random.randn(n_semanas, 3)   # 3 factores latentes
W = np.array([
    [1.0, 0.2, 0.0],
    [0.0, 1.0, 0.3],
    [0.5, 0.0, 1.0],
    [0.8, 0.6, 0.0],   # combinación de las primeras dos
    [0.2, 0.4, 0.9],   # combinación de todas
])
F = base @ W.T   # semanas × métricas (12×5)

print(f'Matriz funnel shape: {F.shape}')
print(f'Rango de la matriz funnel: {np.linalg.matrix_rank(F)}')
print('→ 5 métricas pero solo 3 direcciones verdaderamente independientes')

# Valores singulares revelan la estructura
_, s, _ = np.linalg.svd(F)
print(f'\nValores singulares: {s.round(2)}')
print(f'Energía explicada por cada componente: {(s**2/np.sum(s**2)*100).round(1)}%')

## 3 — Base y dimensión

Una **base** de un espacio vectorial es un conjunto de vectores que:
1. Son **linealmente independientes**.
2. **Abarcan** (span) todo el espacio.

La **dimensión** es el número de vectores en cualquier base. Es única para cada espacio vectorial.

Ejemplo: los vectores estándar $\mathbf{e}_1 = (1,0,\ldots)$, $\mathbf{e}_2 = (0,1,\ldots)$, etc. forman la **base canónica** de $\mathbb{R}^n$.

In [ ]:
# Encontrar una base del espacio columna via SVD
U, s, Vt = np.linalg.svd(F, full_matrices=False)
rank = np.linalg.matrix_rank(F)

# Las primeras `rank` columnas de U forman una base ortonormal del espacio columna
base_col = U[:, :rank]
print(f'Base del espacio columna de F:')
print(f'  Shape: {base_col.shape}  ({n_semanas} semanas, {rank} vectores base)')
print(f'  Los vectores de la base son ortonormales: {np.allclose(base_col.T @ base_col, np.eye(rank))}')

# Espacio nulo via SVD
null_vecs = Vt[rank:].T   # columnas de V correspondientes a sv≈0
print(f'\nEspacio nulo shape: {null_vecs.shape}')
print(f'Verificación F @ null_vec ≈ 0:')
for i in range(null_vecs.shape[1]):
    residuo = np.linalg.norm(F @ null_vecs[:, i])
    print(f'  ||F @ nv{i}|| = {residuo:.2e}')

## 4 — Visualización: rango revela redundancia

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: valores singulares (revelan el rango)
ax = axes[0]
colores = ['#4361ee' if i < rank else '#e8e8e8' for i in range(len(s))]
ax.bar(range(len(s)), s, color=colores, edgecolor='white')
ax.set_xticks(range(len(s)))
ax.set_xticklabels([f'σ{i+1}' for i in range(len(s))])
ax.set_title('Valores singulares de la matriz funnel\n(barras azules = dimensiones con información real)')
ax.set_ylabel('Valor singular')
ax.axhline(1e-10, color='#f72585', linestyle='--', lw=1, label='Umbral rango')
ax.legend()

# Panel 2: energía acumulada
ax2 = axes[1]
energia_cum = np.cumsum(s**2) / np.sum(s**2) * 100
ax2.plot(range(1, len(s)+1), energia_cum, 'o-', color='#4361ee', lw=2, ms=8)
ax2.axhline(99, color='#f72585', linestyle='--', lw=1, label='99% energía')
ax2.fill_between(range(1, rank+1), energia_cum[:rank], alpha=0.2, color='#4361ee')
ax2.set_xlabel('Número de componentes')
ax2.set_ylabel('Energía acumulada (%)')
ax2.set_title('Energía explicada acumulada\n(cuántos componentes capturan la variación real)')
ax2.legend()

plt.tight_layout()
plt.show()

## Resumen

| Concepto | Descripción | NumPy |
|---|---|---|
| Rango | N° de columnas/filas LI | `np.linalg.matrix_rank(A)` |
| LI | Ningún vector es combo lineal de los otros | Verificar rango |
| Espacio columna | Todas las combo lineales de columnas | Cols de U (SVD) |
| Espacio nulo | $\{x: Ax=0\}$ | Cols de V correspondientes a σ≈0 |
| SVD | $A = U\Sigma V^T$, revela estructura | `np.linalg.svd(A)` |
| Nullity-Rank | $\text{rango}+\text{nulidad}=n$ | — |

**Siguiente:** `08_eigenvalues_eigenvectors.ipynb`